# 📘 Notebook 10: Expectation, Variance, Covariance & Correlation

**Week 2 — Calculus, Optimization & Probability Theory**

**Difficulty:** ⭐⭐ (Beginner-Friendly)

---

## 🏠 Why Does This Matter?

A distribution is a full probability function — it describes ALL possible outcomes.  
But often you want a **single number summary**:

- **Expectation:** "On average, how much does a house sell for?" → the center
- **Variance:** "How spread out are house prices?" → the uncertainty
- **Covariance:** "Do larger houses tend to cost more?" → are two variables related?
- **Correlation:** "How STRONGLY are sqft and price related?" → standardized relationship

These appear in:
- **Batch normalization:** subtract mean, divide by standard deviation
- **Bias-variance tradeoff:** overfitting = high variance
- **PCA:** eigenvectors of the covariance matrix
- **Feature selection:** correlation between features and target
- **Multi-head attention:** covariance-like relationships between tokens

---

## Part 1: Expectation (Expected Value)

### Plain English First

If you picked a random house from your dataset every day for 1 million days and recorded the price,  
the **expected value** is the average you'd get.

It's the "center of gravity" of the distribution — the balancing point.

**Discrete:** $E[X] = \sum_x x \cdot P(X=x)$ — weight each outcome by its probability, sum up  
**Continuous:** $E[X] = \int_{-\infty}^{\infty} x \cdot f(x)\, dx$ — same idea, with integral instead of sum

### Key Properties

- **Linearity:** E[aX + b] = a·E[X] + b  (scale and shift)
- **Sum rule:** E[X + Y] = E[X] + E[Y]  (always true!)
- **Independence:** E[XY] = E[X]·E[Y]  (only if independent)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────
# Recreate house dataset
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)
N = 100_000

log_prices = np.random.normal(loc=np.log(300_000), scale=0.5, size=N)
prices     = np.exp(log_prices)
bedrooms   = np.random.choice([1,2,3,4,5,6], size=N, p=[0.05,0.25,0.40,0.20,0.08,0.02])
# sqft positively correlated with price
sqft       = 800 + 300 * np.log(prices/200_000) + 200*np.random.randn(N)
sqft       = np.clip(sqft, 400, 6000)

print("EXPECTED VALUES (averages) for House Market")
print(f"  E[price]    = ${np.mean(prices):,.0f}   (average sale price)")
print(f"  E[sqft]     = {np.mean(sqft):,.0f}    (average square footage)")
print(f"  E[bedrooms] = {np.mean(bedrooms):.2f}    (average bedroom count)")
print()

# ─── Linearity of expectation ─────────────────────────────────────
# Suppose a real estate agent earns 3% commission + $5,000 flat fee
# agent_earnings = 0.03 * price + 5000
# E[earnings] = 0.03 * E[price] + 5000  (linearity!)

commission_rate = 0.03
flat_fee        = 5_000

earnings_direct  = np.mean(commission_rate * prices + flat_fee)
earnings_formula = commission_rate * np.mean(prices) + flat_fee

print("Linearity of Expectation: E[0.03 × price + 5000]")
print(f"  Direct computation:    ${earnings_direct:,.2f}")
print(f"  Via formula E[aX+b]:   ${earnings_formula:,.2f}")
print(f"  Same? {abs(earnings_direct - earnings_formula) < 0.01}  ✓")

# ─── Visual: expected value as center of mass ─────────────────────
plt.figure(figsize=(11, 4))
plt.hist(prices/1000, bins=80, density=True, color='steelblue', alpha=0.7)
plt.axvline(np.mean(prices)/1000,  color='red',    linewidth=3, label=f'Mean = ${np.mean(prices)/1000:.0f}k')
plt.axvline(np.median(prices)/1000,color='orange', linewidth=3, linestyle='--', label=f'Median = ${np.median(prices)/1000:.0f}k')
plt.xlabel('House Price ($thousands)', fontsize=11)
plt.ylabel('Density')
plt.title('House Price Distribution: Mean vs Median\n'
          '(Right-skewed: mean is pulled up by expensive outliers)', fontsize=12)
plt.legend(fontsize=10)
plt.xlim(0, 1500)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Note: Mean > Median for house prices (right skew from luxury homes)")
print("This is why median price is the standard real estate metric!")

## Part 2: Variance and Standard Deviation

### Plain English First

Variance measures **how spread out** values are from the mean.

Imagine two neighborhoods:
- Neighborhood A: all houses around $300k (low variance → predictable)
- Neighborhood B: some $100k houses, some $800k houses (high variance → unpredictable)

Both might have the **same average price**, but Neighborhood B is much harder to predict.

$$\text{Var}(X) = E[(X - E[X])^2] = E[X^2] - (E[X])^2$$

$$\text{Std}(X) = \sqrt{\text{Var}(X)} \quad \text{(same units as X — easier to interpret)}$$

### ML Connections
- **High variance model** = overfitting: fits training data too well, fails on new houses
- **Batch normalization:** normalizes activations to mean=0, variance=1 → stable gradients
- **Variance in gradients** explains why deep networks are hard to train without tricks

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Variance and the bias-variance tradeoff
# ─────────────────────────────────────────────────────────────────

print("VARIANCE of house market variables:")
print(f"  Var(price)    = ${np.var(prices):,.0f}")
print(f"  Std(price)    = ${np.std(prices):,.0f}   (±1 std from mean)")
print(f"  Var(sqft)     = {np.var(sqft):.1f}")
print(f"  Std(sqft)     = {np.std(sqft):.1f} sqft")
print()

# ─── Variance of predictions from multiple models ─────────────────
# Simulate the bias-variance tradeoff
# For a FIXED test house, how much do different trained models disagree?

np.random.seed(42)
N_datasets  = 50     # number of different training sets
N_train     = 20     # small training set → high variance in estimates

true_price    = 350_000
predictions_simple = []   # simple model (high bias, low variance)
predictions_complex= []   # complex model (low bias, high variance)

for _ in range(N_datasets):
    # Sample a small training set
    idx         = np.random.choice(N, N_train, replace=False)
    train_sqft  = sqft[idx]
    train_price = prices[idx]

    # Simple model: just predicts the training set mean
    pred_simple = np.mean(train_price)

    # Complex model: linear regression (can overfit to small data)
    X_tr = np.column_stack([np.ones(N_train), train_sqft])
    # Closed-form least squares: w = (X.T X)^{-1} X.T y
    w    = np.linalg.lstsq(X_tr, train_price, rcond=None)[0]
    pred_complex = w[0] + w[1] * 2000   # predict for a 2000 sqft house

    predictions_simple.append(pred_simple)
    predictions_complex.append(pred_complex)

predictions_simple  = np.array(predictions_simple)
predictions_complex = np.array(predictions_complex)

print("Bias-Variance Tradeoff (50 different training sets, 20 houses each):")
print()
print(f"{'Model':15} | {'Bias (mean-true)':18} | {'Variance (std of preds)':25} | {'MSE (bias²+var)'}")
print("-" * 75)
for name, preds in [('Simple (mean)', predictions_simple), ('Complex (linear)', predictions_complex)]:
    bias     = np.mean(preds) - true_price
    variance = np.var(preds)
    mse      = bias**2 + variance   # MSE = bias² + variance
    print(f"  {name:13} | {bias:18.0f} | {np.std(preds):25.0f} | {mse:.0f}")

# ─── Visualize predictions from each model ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, preds, title, color in [
    (axes[0], predictions_simple,  'Simple Model\n(High Bias, Low Variance)', 'steelblue'),
    (axes[1], predictions_complex, 'Complex Model\n(Low Bias, High Variance)', 'orange'),
]:
    ax.scatter(range(len(preds)), preds/1000, alpha=0.7, color=color, s=40)
    ax.axhline(true_price/1000, color='red', linewidth=2.5, linestyle='--',
               label=f'True price ${true_price/1000:.0f}k')
    ax.axhline(np.mean(preds)/1000, color='black', linewidth=2,
               label=f'Mean prediction ${np.mean(preds)/1000:.0f}k')
    ax.set_xlabel('Training set index')
    ax.set_ylabel('Predicted price ($thousands)')
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Bias-Variance Tradeoff: Variance of Predictions Across Training Sets',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 3: Covariance and Correlation

### Plain English First

**Covariance** measures whether two variables tend to move **in the same direction**.

- **Positive covariance:** When sqft is high, price tends to be high → they move together
- **Negative covariance:** When price is high, age tends to be low (newer = pricier) → opposite
- **Zero covariance:** Knowing one tells you nothing about the other

$$\text{Cov}(X, Y) = E[(X-E[X])(Y-E[Y])] = E[XY] - E[X]E[Y]$$

**Problem with covariance:** its value depends on the scale of X and Y.  
Cov(price, sqft) has units of (dollars × sqft) — hard to interpret!

**Correlation fixes this** by dividing by both standard deviations:
$$\rho(X,Y) = \frac{\text{Cov}(X,Y)}{\text{Std}(X) \cdot \text{Std}(Y)} \in [-1, 1]$$

- `ρ = +1`: perfect positive relationship (knowing sqft perfectly predicts price)
- `ρ = -1`: perfect negative relationship
- `ρ = 0`: no linear relationship

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Covariance and correlation between house features
# ─────────────────────────────────────────────────────────────────

# Add an age feature (older houses tend to be cheaper)
age = np.clip(80 - 20 * np.log(prices/300_000) + 15*np.random.randn(N), 1, 100).astype(int)

features = {'price': prices, 'sqft': sqft, 'bedrooms': bedrooms.astype(float), 'age': age.astype(float)}
feat_names = list(features.keys())
X_matrix = np.column_stack(list(features.values()))   # shape: (N, 4)

# ─── Covariance matrix ───────────────────────────────────────────
cov_matrix  = np.cov(X_matrix.T)     # np.cov expects (features × samples)
corr_matrix = np.corrcoef(X_matrix.T)

print("Covariance Matrix (hard to interpret — units matter):")
print(f"{'':12}", end='')
for name in feat_names: print(f"{name:>14}", end='')
print()
for i, name_i in enumerate(feat_names):
    print(f"{name_i:12}", end='')
    for j in range(len(feat_names)):
        print(f"{cov_matrix[i,j]:14.2e}", end='')
    print()

print()
print("Correlation Matrix (dimensionless, always in [-1, 1] — easy to interpret):")
print(f"{'':12}", end='')
for name in feat_names: print(f"{name:>12}", end='')
print()
for i, name_i in enumerate(feat_names):
    print(f"{name_i:12}", end='')
    for j in range(len(feat_names)):
        val   = corr_matrix[i,j]
        color = '!' if abs(val) > 0.5 and i!=j else ' '
        print(f"  {val:+8.4f}{color} ", end='')
    print()

print()
print("Key findings:")
print(f"  price↔sqft:     ρ = {corr_matrix[0,1]:+.4f}  ← strong positive (bigger = pricier)")
print(f"  price↔bedrooms: ρ = {corr_matrix[0,2]:+.4f}  ← moderate positive")
print(f"  price↔age:      ρ = {corr_matrix[0,3]:+.4f}  ← negative (older = cheaper)")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Visualize: scatter plots at different correlations
# ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Sample 2000 houses for scatter plots (faster rendering)
idx_sample = np.random.choice(N, 2000, replace=False)

pairs = [
    ('price', 'sqft',     'Price vs SqFt'),
    ('price', 'bedrooms', 'Price vs Bedrooms'),
    ('price', 'age',      'Price vs Age'),
    ('sqft',  'bedrooms', 'SqFt vs Bedrooms'),
]

for ax, (fname1, fname2, title) in zip(axes, pairs):
    x_vals = features[fname1][idx_sample]
    y_vals = features[fname2][idx_sample]
    rho    = np.corrcoef(features[fname1], features[fname2])[0,1]

    ax.scatter(x_vals/max(x_vals)*100, y_vals/max(y_vals)*100,   # normalize to 0-100
               alpha=0.2, s=10, color='steelblue')
    ax.set_title(f'{title}\nρ = {rho:+.3f}', fontsize=11)
    ax.set_xlabel(fname1 + ' (normalized)')
    ax.set_ylabel(fname2 + ' (normalized)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Scatter Plots: Visualizing Correlations Between House Features',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Batch Normalization: uses mean and variance
#
# Before feeding data to a neural network, we often normalize each feature:
#   normalized = (value - mean) / std
# This gives each feature: mean ≈ 0, std ≈ 1
#
# Why does this help training?
#   - All features on the same scale → gradients balanced
#   - No feature dominates just because it has larger numbers
#   - Activations stay in a good range for gradient flow
# ─────────────────────────────────────────────────────────────────

# Raw features for 5 houses (each is VERY different in scale)
raw_data = np.column_stack([
    prices[:5],          # ~300,000
    sqft[:5],            # ~1500
    bedrooms[:5],        # ~3
    age[:5],             # ~30
])

# Compute mean and std across the training set (not just 5 houses)
feature_means = np.array([np.mean(prices), np.mean(sqft),
                           np.mean(bedrooms), np.mean(age)])
feature_stds  = np.array([np.std(prices),  np.std(sqft),
                           np.std(bedrooms),  np.std(age)])

# Apply z-score normalization to the 5 houses
normalized_data = (raw_data - feature_means) / feature_stds

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, title in [
    (axes[0], raw_data,        'Before Normalization\n(wildly different scales!)'),
    (axes[1], normalized_data, 'After Normalization\n(all features on same scale)'),
]:
    ax.boxplot(data, labels=['price', 'sqft', 'bedrooms', 'age'], showfliers=True)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Feature value')
    if ax == axes[1]:
        ax.axhline(0, color='red', linestyle='--', alpha=0.5, label='Mean=0')
        ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Feature Normalization: Subtract Mean, Divide by Std\n'
             'Essential for neural network training — prevents any one feature from dominating',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("After normalization:")
print(f"  Mean of each feature: {normalized_data.mean(0).round(3)}  (all ≈ 0)")
print(f"  Std  of each feature: {feature_stds/feature_stds:.3f}  (all = 1)")
print()
print("This is exactly what BatchNorm does inside neural networks!")

---

## ✅ Self-Check Questions

**1.** House prices in City A have mean $300k and std $50k.  
   Prices in City B have mean $300k and std $150k.  
   Which city's prices are easier for your model to predict? Why?

**2.** Var(X + Y) = Var(X) + Var(Y) only if X and Y are independent.  
   If price and sqft are correlated (Cov > 0), is Var(price + sqft) larger or smaller  
   than Var(price) + Var(sqft)?

**3.** The correlation between 'age of house' and 'price' is -0.4.  
   What does this tell you? Is this a strong or weak relationship?

**4.** You compute the covariance matrix of all features. Each diagonal entry is ___?  
   Each off-diagonal entry Cov(Xᵢ, Xⱼ) represents ___?

**5.** Batch normalization subtracts the mean and divides by the standard deviation.  
   After normalization, what are the mean and variance of each feature?

<details>
<summary>Click to see answers</summary>

1. **City A** (std=$50k). Lower variance means prices are more predictable — the typical error in your prediction is only ±$50k. City B has ±$150k uncertainty, making predictions much harder.

2. **Larger**: Var(X+Y) = Var(X) + Var(Y) + **2·Cov(X,Y)**. Since Cov > 0, the sum variance is larger than the sum of individual variances.

3. Older houses tend to be cheaper: for every 1 standard deviation increase in age, price decreases by 0.4 standard deviations. This is a **moderate negative** relationship (|0.4| is between 0 = no relationship and 1 = perfect).

4. Diagonal entries = **Var(Xᵢ)** (each feature's variance). Off-diagonal Cov(Xᵢ,Xⱼ) = **how much features i and j move together** (positive = both increase, negative = one up, other down, zero = unrelated).

5. After batch normalization: mean = **0**, variance = **1** (for each feature). This is the z-score transformation: (x − μ) / σ.

</details>

---

## 📌 Summary

| Concept | Formula | House price meaning | ML connection |
|---------|---------|--------------------|--------------|
| **E[X]** | Σx·P(x) | Average sale price | MSE minimizes expected squared error |
| **Var(X)** | E[(X−μ)²] | Price spread | Model variance in bias-var tradeoff |
| **Std(X)** | √Var(X) | ±1 std price range | Normalization denominator |
| **Cov(X,Y)** | E[(X−μₓ)(Y−μᵧ)] | Do sqft and price move together? | PCA, attention mechanisms |
| **ρ(X,Y)** | Cov/√(Var·Var) | Same, scaled to [-1,1] | Feature selection, correlation analysis |

**➡️ Next Notebook:** Maximum Likelihood Estimation — how all these probability tools combine to explain WHY we use MSE and cross-entropy as loss functions.